In [1]:
import json
import os
from datetime import datetime, timedelta
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import plotly.express as px
import matplotlib.pyplot as plt
from prophet import Prophet

# --- Configuration ---
plt.style.use('seaborn-v0_8')
px.defaults.template = 'plotly_white'

CACHE_DIR = Path('cache')
CACHE_DIR.mkdir(exist_ok=True)

EIA_API_KEY = os.getenv('EIA_API_KEY')
EIA_BASE_URL = 'https://api.eia.gov/v2/electricity/'

# Codes for "Carbon-Free" energy sources
GREEN_CODES = {'SUN', 'WND', 'WAT', 'GEO', 'NUC'}

## EIA Hourly Demand Fetching


In [2]:
# --- 1. Fetch Real Hourly Demand ---
@lru_cache(maxsize=None)
def fetch_eia_hourly(region: str) -> pd.DataFrame:
    """Fetch hourly demand (MW)."""
    url = EIA_BASE_URL + 'rto/region-data/data/'
    end = datetime.utcnow()
    start = end - timedelta(days=90)
    
    params = {
        'api_key': EIA_API_KEY,
        'data[0]': 'value',
        'facets[respondent][]': region,
        'frequency': 'hourly',
        'start': start.strftime('%Y-%m-%dT%H'),
        'end': end.strftime('%Y-%m-%dT%H'),
        'sort[0][column]': 'period',
        'sort[0][direction]': 'desc',
        'length': 5000,
    }
    
    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        data = response.json().get('response', {}).get('data', [])
        df = pd.DataFrame(data)
        if not df.empty:
            df['datetime'] = pd.to_datetime(df['period'])
            df['demand_MW'] = df['value'].astype(float)
            return df.sort_values('datetime')
        return pd.DataFrame()
    except Exception as e:
        print(f"⚠️ Demand fetch failed for {region}: {e}")
        return pd.DataFrame()

# --- 2. Fetch Real Fuel Mix ---
@lru_cache(maxsize=None)
def fetch_eia_fuelmix(region: str) -> float:
    """Fetch Carbon-Free Energy % (0-100)."""
    url = EIA_BASE_URL + 'rto/fuel-type-data/data/'
    end = datetime.utcnow()
    start = end - timedelta(hours=24)
    
    params = {
        'api_key': EIA_API_KEY,
        'data[0]': 'value',
        'facets[respondent][]': region,
        'frequency': 'hourly',
        'start': start.strftime('%Y-%m-%dT%H'),
        'end': end.strftime('%Y-%m-%dT%H'),
        'length': 500
    }
    
    try:
        r = requests.get(url, params=params, timeout=30)
        data = r.json().get('response', {}).get('data', [])
        if not data: return float('nan')
            
        df = pd.DataFrame(data)
        df['value'] = df['value'].astype(float)
        
        # Get most recent hour
        latest_time = df['period'].max()
        current_mix = df[df['period'] == latest_time]
        
        total = current_mix['value'].sum()
        if total == 0: return 0.0
            
        clean = current_mix[current_mix['fueltype'].isin(GREEN_CODES)]['value'].sum()
        return (clean / total) * 100.0
    except:
        return float('nan')

# --- 3. Fetch Real Price (NEW) ---
@lru_cache(maxsize=None)
def fetch_eia_price(region: str) -> float:
    """Fetch latest Wholesale Price ($/MWh)."""
    url = EIA_BASE_URL + 'wholesale-markets-data/data/'
    end = datetime.utcnow()
    start = end - timedelta(hours=24)
    
    # Note: Mapping Regions to Hubs is complex. 
    # For this prototype, we will use a heuristic: 
    # If specific price data is missing, we fallback to a calculated proxy later.
    # This generic call attempts to get any LMP data for the region.
    params = {
        'api_key': EIA_API_KEY,
        'data[0]': 'value',
        'facets[respondent][]': region,
        'frequency': 'hourly',
        'sort[0][column]': 'period',
        'sort[0][direction]': 'desc',
        'length': 10
    }
    try:
        r = requests.get(url, params=params, timeout=5)
        data = r.json().get('response', {}).get('data', [])
        if data:
            # return average of the last few reported prices
            vals = [float(x['value']) for x in data if x['value']]
            return np.mean(vals) if vals else float('nan')
    except:
        pass
    return float('nan')

# --- 4. Fetch Weather ---
@lru_cache(maxsize=None)
def fetch_temperature(lat, lon):
    try:
        url = 'https://api.open-meteo.com/v1/forecast'
        params = {'latitude': lat, 'longitude': lon, 'daily': 'temperature_2m_mean', 'past_days': 60}
        r = requests.get(url, params=params, timeout=10)
        temps = r.json().get('daily', {}).get('temperature_2m_mean', [])
        return np.mean(temps) if temps else np.nan
    except:
        return np.nan

## Compute Datacenter Scores

In [5]:

# --- MAIN EXECUTION ---
region_coords = {
    'CAL': (36.5, -119.5), 'CAR': (35.5, -80.0), 'CENT': (38.5, -94.5),
    'FLA': (28.0, -82.0), 'MIDA': (39.0, -77.0), 'MIDW': (42.0, -89.0),
    'NE': (42.5, -72.5), 'NY': (42.9, -75.3), 'NW': (45.5, -120.5),
    'SE': (33.0, -84.0), 'SW': (36.0, -111.5), 'TEN': (36.0, -86.0),
    'TEX': (31.0, -99.0),
}

records = []
print("🚀 Fetching Real Data...")

for region, (lat, lon) in region_coords.items():
    # A. Fetch Raw Data
    df_demand = fetch_eia_hourly(region)
    raw_renew = fetch_eia_fuelmix(region)
    raw_price = fetch_eia_price(region)
    raw_temp = fetch_temperature(lat, lon)
    
    # B. Compute Derived Metrics
    if not df_demand.empty:
        raw_load = df_demand['demand_MW'].iloc[-1]
        # Volatility (Standard Deviation of last 24h)
        raw_volatility = df_demand['demand_MW'].tail(24).std()
        # Peak Forecast (Simple max of last 30 days as a proxy for capacity)
        raw_peak = df_demand['demand_MW'].max()
    else:
        raw_load, raw_volatility, raw_peak = np.nan, np.nan, np.nan

    # Fallback logic if Price API returns nothing (common for some regions)
    # We proxy price using Load Stress (High Load = High Price)
    if np.isnan(raw_price) and not np.isnan(raw_load):
        raw_price = (raw_load / 1000) * 2.5 # Rough heuristic $2.50 per GW

    records.append({
        'region': region,
        'lat': lat, 'lon': lon,
        'raw_price': raw_price,
        'raw_load': raw_load,
        'raw_volatility': raw_volatility,
        'raw_peak': raw_peak,
        'raw_renew': raw_renew,
        'raw_temp': raw_temp
    })

# --- DATAFRAME CONSTRUCTION ---
dc_df = pd.DataFrame(records)

# 1. Fill Missing Data (Mean Imputation)
dc_df = dc_df.fillna(dc_df.mean(numeric_only=True))

# 2. Normalize Columns (Store as 'n_column')
# We normalize so 0 is "Bad" and 1 is "Good"
# Price: Lower is better -> 1 - norm
# Load: Lower is better -> 1 - norm
# Volatility: Lower is better -> 1 - norm
# Temp: Lower is better -> 1 - norm
# Renewables: Higher is better -> norm

def normalize(series, invert=False):
    min_v, max_v = series.min(), series.max()
    if max_v == min_v: return 0.5
    norm = (series - min_v) / (max_v - min_v)
    return (1 - norm) if invert else norm

dc_df['n_price'] = normalize(dc_df['raw_price'], invert=True)
dc_df['n_load'] = normalize(dc_df['raw_load'], invert=True)
dc_df['n_volatility'] = normalize(dc_df['raw_volatility'], invert=True)
dc_df['n_temp'] = normalize(dc_df['raw_temp'], invert=True)
dc_df['n_renew'] = normalize(dc_df['raw_renew'], invert=False) # Higher is good

# 3. Calculate Scores
# Profitability (40%): Price, Load, Volatility
dc_df['profitability'] = (
    0.40 * dc_df['n_price'] +
    0.30 * dc_df['n_load'] +
    0.30 * dc_df['n_volatility']
)

# Sustainability (60%): Renewables, Temp
dc_df['sustainability'] = (
    0.70 * dc_df['n_renew'] + 
    0.30 * dc_df['n_temp']
)

# Final Score
dc_df['dc_score'] = 0.40 * dc_df['profitability'] + 0.60 * dc_df['sustainability']

# Sort
dc_df_final = dc_df.sort_values('dc_score', ascending=False).reset_index(drop=True)

# Save EVERYTHING
dc_df_final.to_csv('datacenter_scores_real.csv', index=False)

🚀 Fetching Real Data...


In [6]:
dc_df_final

,region,lat,lon,raw_price,raw_load,raw_volatility,raw_peak,raw_renew,raw_temp,n_price,n_load,n_volatility,n_temp,n_renew,profitability,sustainability,dc_score
0,CAR,35.5,-80.0,53.1725,21269.0,1162.005039,35825.0,62.274280,14.841791,0.879192,0.879192,0.888992,0.458757,1.000000,0.882132,0.837627,0.855429
1,NY,42.9,-75.3,44.5450,17818.0,973.780909,20325.0,46.294596,6.732836,0.922797,0.922797,0.948533,1.000000,0.658137,0.930518,0.760696,0.828625
2,TEN,36.0,-86.0,43.0200,17208.0,847.219620,26174.0,48.673019,14.868657,0.930505,0.930505,0.988569,0.456964,0.709020,0.947924,0.633403,0.759211
3,NW,45.5,-120.5,92.6250,37050.0,1417.739076,47886.0,49.890434,9.132836,0.679791,0.679791,0.808095,0.839809,0.735065,0.718282,0.766488,0.747206
4,CENT,38.5,-94.5,77.0525,30821.0,1724.997038,48354.0,49.616232,14.391045,0.758497,0.758497,0.710900,0.488842,0.729198,0.744218,0.657092,0.691942
5,NE,42.5,-72.5,36.5100,14604.0,903.658433,16897.0,31.877865,8.579104,0.963408,0.963408,0.970715,0.876768,0.349711,0.965600,0.507828,0.690937
6,SW,36.0,-111.5,29.2700,11708.0,811.082761,20834.0,25.881175,12.679104,1.000000,1.000000,1.000000,0.603108,0.221420,1.000000,0.335926,0.601556
7,SE,33.0,-84.0,64.2350,25694.0,1597.993973,38910.0,34.138589,16.273134,0.823280,0.823280,0.751075,0.363220,0.398076,0.801618,0.387619,0.553219
8,CAL,36.5,-119.5,63.4525,25381.0,1629.419288,38450.0,27.187905,16.710448,0.827235,0.827235,0.741134,0.334031,0.249375,0.801404,0.274772,0.485425
9,MIDA,39.0,-77.0,227.1250,90850.0,2685.051468,111773.0,38.825942,12.791045,0.000000,0.000000,0.407204,0.595637,0.498355,0.122161,0.527540,0.365388
